# PersonaPlex Launcher — RTX 5090 / A100

Goal command (runs last, after setup):

```
%cd personaplex
!python -m moshi_local.moshi.server --web-search-enabled --no-compressor-4bit --checkpoint-dir moshi_local/lora --rag-index-dir rag_index
```

Every cell above the launch section exists only to make that command work — installing exactly what `moshi_local/moshi/server.py` imports, nothing more. The PyTorch build (`cu128`) supports both Blackwell (RTX 5090) and Ampere (A100), so this notebook runs unchanged on either GPU.

In [ ]:
import os

if not os.path.isdir("personaplex"):
    !git clone https://github.com/MoshiHead/original-s-sytem-run-26-july.git personaplex
else:
    print("personaplex/ already present, skipping clone")

In [ ]:
!nvidia-smi

In [ ]:
!apt-get update -qq && apt-get install -y -qq libopus-dev

In [ ]:
# Fresh, matched versions avoid a base-image torch build that predates Blackwell (sm_120) support.
!pip uninstall -y -q torch torchvision torchaudio transformers sentence-transformers sentencepiece accelerate safetensors

In [ ]:
!pip install torch==2.8.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128

In [ ]:
!pip install "numpy>=1.26,<2.2" "einops==0.7" "sphn>=0.1.4,<0.2" "aiohttp>=3.10.5,<3.11" "sentencepiece==0.2.0" "safetensors>=0.4.0,<0.5" "transformers==4.52.4" "sentence-transformers==4.1.0" accelerate huggingface_hub peft faiss-cpu

In [ ]:
# --no-deps: the moshi package on PyPI wants sphn>=0.2, which conflicts with the
# sphn<0.2 that moshi_local (this fork) requires. We already installed the sphn we need above.
!pip install --no-deps moshi

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Compute capability:", torch.cuda.get_device_capability(0))

In [ ]:
import os
from getpass import getpass

from huggingface_hub import login

# hf_token = os.environ.get("HF_TOKEN")
# sa token...not mine
hf_token = 'tLNSyNjFduNaLUbvyxosVqiGwuAtiQPOTt'    
if not hf_token:
    hf_token = getpass("Enter your Hugging Face access token (input hidden): ")

os.environ["HF_TOKEN"] = hf_token
login(token=hf_token, add_to_git_credential=False)
print("Logged in to Hugging Face Hub.")

## Download PersonaPlex model assets

Pre-fetches the gated `nvidia/personaplex-7b-v1` files (config, tokenizer, Mimi codec weights, Moshi LM weights, voice prompts, web UI) into the Hugging Face cache. `server.py` would fetch these lazily on first use anyway — doing it here just surfaces a license/token problem now instead of mid-startup, and warms the cache before the server needs it.

In [ ]:
from huggingface_hub import hf_hub_download

HF_REPO_ID = "nvidia/personaplex-7b-v1"
ASSET_FILES = [
    "config.json",
    "tokenizer_spm_32k_3.model",
    "tokenizer-e351c8d8-checkpoint125.safetensors",
    "model.safetensors",
    "voices.tgz",
    "dist.tgz",
]

downloaded = {}
try:
    for fname in ASSET_FILES:
        path = hf_hub_download(HF_REPO_ID, fname)
        downloaded[fname] = path
        print(f"OK  {fname} -> {path}")
except Exception as e:
    raise RuntimeError(
        "Failed to download model assets from "
        f"https://huggingface.co/{HF_REPO_ID}. This almost always means either:\n"
        "  1) you have not clicked 'Agree and access repository' on that model page yet, or\n"
        "  2) the HF_TOKEN you supplied doesn't belong to the account that accepted the license, or\n"
        "  3) the token is invalid/expired.\n"
        f"Original error: {e}"
    )

In [ ]:
import pathlib
import tarfile

for tgz_name in ("voices.tgz", "dist.tgz"):
    tgz_path = pathlib.Path(downloaded[tgz_name])
    out_dir = tgz_path.parent / tgz_name.replace(".tgz", "")
    already = out_dir.exists()
    if not already:
        with tarfile.open(tgz_path, "r:gz") as tar:
            tar.extractall(path=tgz_path.parent)
    print(f"{tgz_name} -> {out_dir} ({'already extracted' if already else 'extracted now'})")

In [ ]:
import json
import os

from huggingface_hub import hf_hub_download

lora_dir = "personaplex/moshi_local/lora"
os.makedirs(lora_dir, exist_ok=True)

# adapter_config.json isn't in the Darknsu/helium_lora_v1 dataset repo (it only has
# adapter_model.safetensors) — writing the matching config directly instead of fetching it.
# This is the same config that ships with the local moshi_local/lora checkpoint; the remote
# adapter_model.safetensors is byte-identical in size (187706112 bytes), so it should be the
# same run. If you retrain, update this dict (or add adapter_config.json to the HF repo instead).
adapter_config = {
    "alora_invocation_tokens": None,
    "alpha_pattern": {},
    "arrow_config": None,
    "auto_mapping": None,
    "base_model_name_or_path": None,
    "bias": "none",
    "corda_config": None,
    "ensure_weight_tying": False,
    "eva_config": None,
    "exclude_modules": None,
    "fan_in_fan_out": False,
    "inference_mode": True,
    "init_lora_weights": True,
    "layer_replication": None,
    "layers_pattern": None,
    "layers_to_transform": None,
    "loftq_config": {},
    "lora_alpha": 256.0,
    "lora_bias": False,
    "lora_dropout": 0.05,
    "lora_ga_config": None,
    "megatron_config": None,
    "megatron_core": "megatron.core",
    "modules_to_save": None,
    "peft_type": "LORA",
    "peft_version": "0.19.1",
    "qalora_group_size": 16,
    "r": 128,
    "rank_pattern": {},
    "revision": None,
    "target_modules": ["proj", "fc1", "out_proj", "fc2", "linear", "in_proj"],
    "target_parameters": None,
    "task_type": "FEATURE_EXTRACTION",
    "trainable_token_indices": None,
    "use_bdlora": None,
    "use_dora": False,
    "use_qalora": False,
    "use_rslora": False,
}
with open(os.path.join(lora_dir, "adapter_config.json"), "w") as f:
    json.dump(adapter_config, f, indent=2)

weights_path = os.path.join(lora_dir, "adapter_model.safetensors")
if not os.path.exists(weights_path):
    print("Downloading adapter_model.safetensors ...")
    hf_hub_download(
        repo_id="Darknsu/helium_lora_v1",
        repo_type="dataset",
        filename="adapter_model.safetensors",
        local_dir=lora_dir,
    )
else:
    print("adapter_model.safetensors already present, skipping download")

print("LoRA checkpoint ready:", os.listdir(lora_dir))

## Launch

Ensures `rag_index` exists next to `personaplex` (copied from the repo-root `rag_index` if needed), then runs the target command.

In [ ]:
%cd personaplex

In [ ]:
import os, shutil

if not os.path.isdir("rag_index") and os.path.isdir("../rag_index"):
    shutil.copytree("../rag_index", "rag_index")

assert os.path.isdir("rag_index"), "rag_index not found"
assert os.path.isdir("moshi_local/lora"), "LoRA checkpoint not found at moshi_local/lora"
print("rag_index and checkpoint ready")

In [ ]:
!python -m moshi_local.moshi.server --web-search-enabled --no-compressor-4bit --checkpoint-dir moshi_local/lora --rag-index-dir rag_index